In [1]:
import numpy as np 
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
featured_data =pd.read_csv("C:/Users/neilg/Desktop/New_folder/vs_code/ais/data/gulf_features.csv")

In [3]:
featured_data.head(10)

,mmsi,base_datetime,ping_sequence,lat,lon,cog,heading,sog,time_differ,dist
0,83560169,2023-01-19 15:40:12,1,29.54868,-89.77021,253.3,315,3.2,NaN,NaN
1,209130856,2023-01-07 00:12:25,1,29.46187,-89.59770,186.1,448,0.0,NaN,NaN
2,209497000,2023-01-15 17:54:50,1,27.50247,-88.57090,332.0,333,13.0,NaN,NaN
3,209497000,2023-01-15 17:57:20,2,27.51030,-88.57557,332.0,333,12.9,2.500000,0.984976
4,209497000,2023-01-15 17:58:30,3,27.51397,-88.57775,331.0,333,12.9,1.166667,0.461254
5,209497000,2023-01-15 17:59:40,4,27.51782,-88.58002,332.0,333,12.9,1.166667,0.483098
6,209497000,2023-01-15 18:00:40,5,27.52097,-88.58187,331.0,333,12.9,1.000000,0.394927
7,209497000,2023-01-15 18:05:51,6,27.53758,-88.59157,332.0,333,12.9,5.183333,2.079915
8,209497000,2023-01-15 18:11:40,7,27.55633,-88.60222,333.0,333,12.9,5.816667,2.334367
9,209497000,2023-01-15 18:12:51,8,27.56017,-88.60447,333.0,333,12.9,1.183333,0.481161


In [4]:
print(featured_data.shape)
print(featured_data.dtypes)
print(featured_data.isnull().sum())

(5185360, 10)
mmsi               int64
base_datetime     object
ping_sequence      int64
lat              float64
lon              float64
cog              float64
heading            int64
sog              float64
time_differ      float64
dist             float64
dtype: object
mmsi               0
base_datetime      0
ping_sequence      0
lat                0
lon                0
cog                0
heading            0
sog                0
time_differ      300
dist             300
dtype: int64


In [5]:
featured_data['base_datetime'] = pd.to_datetime(featured_data['base_datetime'])
print(featured_data.dtypes['base_datetime'])

datetime64[ns]


In [6]:
featured_data.groupby('mmsi').size().sort_values()

mmsi
83560169          1
209130856         1
321687710         1
338196992         1
333950458         1
              ...  
367300080    105386
366997820    105556
366993660    106625
368011470    107120
367305890    107788
Length: 300, dtype: int64

In [7]:
featured_data = featured_data.dropna()

In [8]:
print(featured_data.isnull().sum())

mmsi             0
base_datetime    0
ping_sequence    0
lat              0
lon              0
cog              0
heading          0
sog              0
time_differ      0
dist             0
dtype: int64


In [9]:
print(featured_data.shape)

(5185060, 10)


In [10]:
featured_data.groupby('mmsi').size()

mmsi
209497000    2223
209961000    7337
210296000      36
212287000     569
229214000    6619
             ... 
636022337    1583
636022536    2444
636092848    1074
636093086     730
941217433       5
Length: 278, dtype: int64

In [21]:
featured_data['dist_log'] = np.log1p(featured_data['dist'])
featured_data['time_differ_log'] = np.log1p(featured_data['time_differ'])

scalar = StandardScaler()

In [22]:
raw_data_headings= ['lat','lon','cog','sog','time_differ_log','dist_log']


In [23]:
data_scaled_headings= ['lat_scaled','lon_scaled','cog_scaled','sog_scaled','time_differ_scaled','dist_scaled']


In [24]:
featured_data[data_scaled_headings]=scalar.fit_transform(featured_data[raw_data_headings])

In [25]:
featured_data = featured_data.reset_index(drop=True)

In [19]:
featured_data.index

RangeIndex(start=0, stop=5185060, step=1)

In [26]:
sequence_length = 50
feature_cols = data_scaled_headings 
print(feature_cols)
print(type(feature_cols))


['lat_scaled', 'lon_scaled', 'cog_scaled', 'sog_scaled', 'time_differ_scaled', 'dist_scaled']
<class 'list'>


In [27]:
test_group = featured_data[featured_data['mmsi'] == 209497000]
test_chunk = test_group[feature_cols].iloc[0:50]
print(test_chunk.shape)

(50, 6)


In [28]:
sequence_length = 50
feature_cols = data_scaled_headings

all_sequences = []
for mmsi, group in featured_data.groupby('mmsi'):
    group = group.sort_values('base_datetime')
    
    group_array = np.array(group[feature_cols])              
    
    n_rows = len(group_array)              
    n_full_chunks = n_rows // sequence_length
    
    for i in range(n_full_chunks):
        start = i * sequence_length
        end = start + sequence_length
        chunk = group_array[start:end]     
        all_sequences.append(chunk)

X = np.array(all_sequences)
print(X.shape)

(103566, 50, 6)


In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping


In [31]:


model = Sequential([
    LSTM( 32, activation='relu', input_shape=(50, 6)),   
    RepeatVector(50),                                        # repeat bottleneck across timesteps
    LSTM(32, activation='relu', return_sequences=True),        # decoder
    TimeDistributed(Dense(6))                                 # map back to feature count per timestep
])

from tensorflow.keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse')
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         4,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 50, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 50, 6)          │           198 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,510 (52.77 KB)

 Trainable params: 13,510 (52.77 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X, X,
    epochs=200,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - loss: 14.0846 - val_loss: 264.5215
Epoch 2/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 31s 21ms/step - loss: 16.2668 - val_loss: 17.1202
Epoch 3/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 29s 20ms/step - loss: 15.8845 - val_loss: 8.8541
Epoch 4/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 29s 20ms/step - loss: 11.5325 - val_loss: 9.3669
Epoch 5/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 28s 19ms/step - loss: 96.3047 - val_loss: 8.1867
Epoch 6/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 32s 22ms/step - loss: 4.1603 - val_loss: 2.2529
Epoch 7/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 34s 23ms/step - loss: 1.5307 - val_loss: 2.1346
Epoch 8/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 31s 22ms/step - loss: 1.8014 - val_loss: 3.2256
Epoch 9/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 32s 22ms/step - loss: 2.8527 - val_loss: 1.4919
Epoch 10/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 32s 22ms/step - loss: 0.9866 - val_loss: 1.7203
Epoch 11/200
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 33s 22ms/step - loss: 1.7871 - val_lo

In [36]:
x_restructured=model.predict(X)

3237/3237 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step


In [34]:
print(featured_data['dist'].describe())
print(featured_data['time_differ'].describe())

count    5.185060e+06
mean     8.703879e-02
std      9.356372e-01
min      0.000000e+00
25%      1.471329e-03
50%      2.947741e-03
75%      3.326573e-02
max      3.619015e+02
Name: dist, dtype: float64
count    5.185060e+06
mean     3.253084e+00
std      2.027880e+02
min      1.666667e-02
25%      1.150000e+00
50%      1.166667e+00
75%      1.350000e+00
max      1.228961e+05
Name: time_differ, dtype: float64
